In [1]:
import math


import torch
import torch.nn as nn
import torch.nn.functional as F


import torchvision
from torchvision import transforms

In [2]:
train_dataset = torchvision.datasets.ImageFolder(
    "datasets/butterflies",
    transform=transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ]
    ),
)

In [3]:
class PatchEmbed(nn.Module):
    """ """

    def __init__(
        self,
        img_size: int | tuple[int, int] = 224,
        patch_size: int | tuple[int, int] = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        bias: bool = True,
    ):
        super().__init__()

        if isinstance(img_size, int):
            img_size = (img_size, img_size)
        if isinstance(patch_size, int):
            patch_size = (patch_size, patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = tuple(
            [
                s // p
                for s, p in zip(
                    img_size,
                    patch_size,
                )
            ]
        )

        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=bias,
        )

    def forward(self, x: torch.Tensor):
        # (B, C, H, W) <= x

        # B: batch
        # C: channel
        # H: height (original)
        # W: width (original)

        # D: embed_dim
        # h: height (projected)
        # w: width (projected)
        # L: seqlen = h*w

        # (B, D, h, w)
        x = self.proj(x)

        # (B, D, L)
        # (B, L, D)
        x = x.flatten(2).transpose(1, 2)

        return x


In [4]:
patch_embed = PatchEmbed()

img_tensor, label = train_dataset[0]
image_batch = img_tensor[None, :]
print(image_batch.shape)
image_projected = patch_embed(image_batch)
print(image_projected.shape)
print(patch_embed.grid_size)
print(patch_embed.num_patches)

torch.Size([1, 3, 224, 224])
torch.Size([1, 196, 768])
(14, 14)
196


In [5]:
class TimestepEmbedder(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        frequency_embedding_size: int = 256,
        use_bias=True,
    ):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=use_bias),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=use_bias),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(
        t: torch.Tensor,
        dim: int,
        max_period: int = 10000,
        freq_scale: int = 1,
    ):
        assert dim % 2 == 0

        half = dim // 2
        freqs = freq_scale * torch.exp(
            #####
            -math.log(max_period) * torch.arange(0, half, dtype=torch.float32) / half
        ).to(
            device=t.device,
        )

        #     t: (B,)
        # freqs: (half,)
        #  args: (B, half)
        args = t[:, None].float() * freqs[None, :]
        # embedding: (B, dim)
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return embedding

    def forward(
        self,
        t: torch.Tensor,
        freq_scale: int = 1,
    ):
        #      t: (B,)
        # t_freq: (B, frequency_embedding_size)
        #  t_emb: (B, D)
        t_freq = self.timestep_embedding(
            t, self.frequency_embedding_size, freq_scale=freq_scale
        )
        t_emb = self.mlp(t_freq)
        return t_emb


In [6]:
timestep_embedder = TimestepEmbedder(768)
timestep_embedder(
    torch.tensor(
        [0, 100, 900],
    )
)

tensor([[-0.0497, -0.0509, -0.0946,  ..., -0.0998, -0.1481,  0.2758],
        [-0.0025,  0.0466,  0.0931,  ...,  0.0184, -0.1260,  0.1613],
        [ 0.0545,  0.0242,  0.1953,  ..., -0.1148, -0.1282, -0.0480]],
       grad_fn=<AddmmBackward0>)

In [7]:
class LabelEmbedder(nn.Module):
    def __init__(
        self,
        num_classes: int,
        hidden_size: int,
        dropout_prob: float,
    ):
        super().__init__()
        use_cfg_embedding = 1 if (dropout_prob > 0.0) else 0
        self.embedding_table = nn.Embedding(
            num_classes + use_cfg_embedding,
            hidden_size,
        )
        self.num_classes = num_classes
        self.dropout_prob = dropout_prob

    def token_drop(
        self,
        labels: torch.Tensor,
        force_drop_mask: torch.Tensor = None,
    ):
        # force_drop_mask: 1D mask to labels
        if force_drop_mask is None:
            drop_mask = (
                torch.rand(labels.shape[0], device=labels.device) < self.dropout_prob
            )
        else:
            assert force_drop_mask.shape == labels.shape
            drop_mask = force_drop_mask.bool()

        labels = torch.where(drop_mask, self.num_classes, labels)
        return labels

    def forward(
        self,
        labels: torch.Tensor,
        train: bool,
        force_drop_mask: torch.Tensor = None,
    ):
        use_dropout = (
            #####
            self.dropout_prob > 0
        )
        if (train and use_dropout) or (force_drop_mask is not None):
            labels = self.token_drop(
                labels=labels,
                force_drop_mask=force_drop_mask,
            )
        embeddings = self.embedding_table(labels)
        return embeddings


In [8]:
label_embedder = LabelEmbedder(
    num_classes=20,
    hidden_size=768,
    dropout_prob=0.8,
)
label_embedder(
    torch.arange(0, 20),
    train=True,
    # force_drop_mask=torch.tensor([True] * 20),
)

tensor([[-0.0376,  0.2116, -0.0907,  ..., -0.7235,  0.1159,  0.1635],
        [-0.0376,  0.2116, -0.0907,  ..., -0.7235,  0.1159,  0.1635],
        [-0.0376,  0.2116, -0.0907,  ..., -0.7235,  0.1159,  0.1635],
        ...,
        [-0.0376,  0.2116, -0.0907,  ..., -0.7235,  0.1159,  0.1635],
        [-0.0376,  0.2116, -0.0907,  ..., -0.7235,  0.1159,  0.1635],
        [-0.6602, -0.2730,  0.1970,  ..., -0.9349, -2.2518, -1.1035]],
       grad_fn=<EmbeddingBackward0>)

In [9]:
class Attention(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        n_heads: int = 8,
        qkv_bias: bool = True,
        proj_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ):
        super().__init__()
        assert embed_dim % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.scale = self.head_dim ** (-0.5)

        self.w_qkv = nn.Linear(embed_dim, embed_dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(embed_dim, embed_dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, embed_dim = x.shape

        # (B, L, embed_dim)
        # (B, L, 3, n_heads, head_dim)
        # (3, B, n_heads, L, head_dim)
        qkv: torch.Tensor = (
            self.w_qkv(x)
            .reshape(B, L, 3, self.n_heads, self.head_dim)
            .permute(2, 0, 3, 1, 4)
        )

        # (B, n_heads, L, head_dim)
        q, k, v = qkv.unbind(0)
        x = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=(self.attn_drop.p if self.training else 0.0),
        )
        x = x.transpose(1, 2).reshape(B, L, embed_dim)

        x = self.proj(x)
        x = self.proj_drop(x)

        return x


class Mlp(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: int = None,
        out_features: int = None,
        bias: bool | tuple[bool, bool] = True,
        drop_probs: float | tuple[float, float] = 0.0,
    ):
        super().__init__()

        if hidden_features is None:
            hidden_features = in_features

        if out_features is None:
            out_features = in_features

        if isinstance(bias, bool):
            bias = (bias, bias)

        if isinstance(drop_probs, float):
            drop_probs = (drop_probs, drop_probs)

        self.fc1 = nn.Linear(in_features, hidden_features, bias=bias[0])
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(drop_probs[0])
        self.fc2 = nn.Linear(hidden_features, out_features, bias=bias[1])
        self.drop2 = nn.Dropout(drop_probs[1])

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


In [10]:
def modulate(x, shift, scale):
    #     x: (B, L, D)
    # shift: (B, D)
    # scale: (B, D)
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTBlock(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        n_heads: int,
        mlp_ratio: float = 4.0,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = Attention(hidden_size, n_heads=n_heads)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)

        mlp_hidden_dim = int(hidden_size * mlp_ratio)
        self.mlp = Mlp(
            in_features=hidden_size,
            hidden_features=mlp_hidden_dim,
            drop_probs=0.0,
        )

        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True),
        )

        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(
        self,
        x: torch.Tensor,
        c: torch.Tensor,
    ):
        # x: (B, L, D)
        # c: (B, D)

        # (B, D)
        (
            shift_msa,
            scale_msa,
            gate_msa,
            #####
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.adaLN_modulation(c).chunk(6, dim=1)

        x = x + gate_msa[:, None] * self.attn(
            modulate(self.norm1(x), shift_msa, scale_msa)
        )
        x = x + gate_mlp[:, None] * self.mlp(
            modulate(self.norm2(x), shift_mlp, scale_mlp)
        )

        return x
